# 01 · Healthy lung atlas: donor filtering and doublet detection

First step of the healthy-lung pipeline (paper Methods: *Non-malignant lung single cell data processing*).

**Input**: Human Lung Cell Atlas (HLCA) core, Sikkema *et al.* 2023 (CELLxGENE `.h5ad`)

**Steps**
1. Keep donors with ≥ 100 cells and switch to the raw count matrix.
2. Score doublets with Scrublet (`sc.pp.scrublet`), using `donor_id` as the batch key and 15 principal components.

**Output**: `intermediate_files/lung/01_lung_sikkema_atlas_raw_normal.h5ad` (raw counts and doublet scores). Doublets are removed in File 02.

## Setup

In [1]:
import os
import gc

import anndata
import scanpy as sc

In [2]:
# Paths: edit to match your environment
SIKKEMA_H5AD = "/path/to/Data_Sikkema_2023_lung/2b415299-371a-4e95-8cba-8c6036a72ad5.h5ad"  # HLCA core from CELLxGENE
OUT_DIR = "intermediate_files/lung"

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load the atlas and keep donors with ≥ 100 cells

In [3]:
adata = sc.read(SIKKEMA_H5AD)

sample_counts = adata.obs['donor_id'].value_counts()               # number of cells per donor
valid_samples = sample_counts[sample_counts >= 100].index          # donors with at least 100 cells
adata = adata[adata.obs['donor_id'].isin(valid_samples)].copy()

adata = anndata.AnnData(adata.raw.X, obs=adata.obs, var=adata.raw.var)  # use the raw count matrix
gc.collect()

3215

In [4]:
adata

AnnData object with n_obs × n_vars = 584889 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_1', '

In [5]:
adata.var

,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000000003,TSPAN6_ENSG00000000003,NCBITaxon:9606,gene,2396,protein_coding
ENSG00000000005,TNMD_ENSG00000000005,NCBITaxon:9606,gene,873,protein_coding
ENSG00000000419,DPM1_ENSG00000000419,NCBITaxon:9606,gene,1262,protein_coding
ENSG00000000457,SCYL3_ENSG00000000457,NCBITaxon:9606,gene,2916,protein_coding
ENSG00000000460,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...
ENSG00000283078,ENSG00000283078.1,NCBITaxon:9606,gene,1608,lncRNA
ENSG00000283103,ENSG00000283103.5,NCBITaxon:9606,gene,2285,lncRNA
ENSG00000283117,MGC4859,NCBITaxon:9606,gene,3118,lncRNA
ENSG00000283118,ENSG00000283118.1,NCBITaxon:9606,gene,644,lncRNA


## 2. Doublet detection with Scrublet

In [6]:
sc.pp.scrublet(adata, batch_key="donor_id", n_prin_comps=15)  # doublet scores per donor
gc.collect()

## 3. Save

In [7]:
adata.write(f'{OUT_DIR}/01_lung_sikkema_atlas_raw_normal.h5ad')